# exp-back — worked example 1: exp_back Using the Cached out Array

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `exp-back`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The derivative of `exp(x)` with respect to `x` is `exp(x)` itself — the exponential is its own derivative. In a backward pass, the forward output `out = exp(x)` is typically cached during the forward pass so the backward function can reuse it without recomputing `exp(x)`. The chain rule gives `dL/dx = grad_out * out`, where `grad_out` is the upstream gradient.

## Worked solution

We implement `exp_back` and verify it against PyTorch's autograd.

**Step 1 — mathematical derivation:** `y = exp(x)`, so `dy/dx = exp(x) = y`. Chain rule: `dL/dx = dL/dy * dy/dx = grad_out * y = grad_out * out`.

**Step 2 — why reuse `out` instead of recomputing:** Calling `exp(x)` inside the backward function is redundant because the forward pass already computed it. Caching the output avoids this extra computation. It's also why the backward signature accepts both `out` and `x` — `out` is passed specifically so implementations can use it without touching `x`.

**Step 3 — verification:** We test against `torch.autograd` by computing `exp(x).sum().backward()` and checking that our `exp_back(ones_like(out), out, x)` matches `x.grad`.

In [ ]:
import torch as t
from torch import Tensor

t.manual_seed(8)

def exp_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    """Backward of exp(x): dL/dx = grad_out * out. Use cached out, not x."""
    return grad_out * out

# Test on a 1-D tensor
x = t.tensor([0.0, 1.0, -1.0, 2.0])
out = t.exp(x)
grad_out = t.ones_like(out)  # seed from a sum loss

my_grad = exp_back(grad_out, out, x)
print(f"x:       {x.tolist()}")
print(f"out:     {out.tolist()}")
print(f"my_grad: {my_grad.tolist()}")

# Compare to autograd
x2 = x.clone().requires_grad_(True)
t.exp(x2).sum().backward()
print(f"torch:   {x2.grad.tolist()}")
assert t.allclose(my_grad, x2.grad), "Mismatch with torch.autograd"

# Non-unit seed
x3 = t.tensor([1.0, 2.0, 3.0])
out3 = t.exp(x3)
grad_seed = t.tensor([2.0, 0.5, 3.0])
my_grad3 = exp_back(grad_seed, out3, x3)

x4 = x3.clone().requires_grad_(True)
(grad_seed * t.exp(x4)).sum().backward()
assert t.allclose(my_grad3, x4.grad), "Mismatch with weighted seed"
print("exp_back correct for both unit and weighted seeds.")